### Varför behöver vi Airflow?

Airflow är ett orkestreringsverktyg/framework.

Ordet du ska komma ihåg är: **Orkestrering**

Orkestrering betyder ungefär:

att koordinera och styra flera olika processer så att de körs på rätt sätt och i rätt ordning.

**"Vad används Airflow till?"**

Airflow används för att orkestrera och schemalägga data pipelines och hantera beroenden mellan olika steg.

**Vad är Cron?**

Cron är ett traditionellt Unix/Linux-system för att säga:"Kör det här programmet vid den här tiden."

Schemalägger program vid bestämda tider

Exempel:

Varje dag kl. 02:00 -> kör script.py

Du kan alltså schemalägga saker. Till exempel:

- 02:00 → script A
- 03:00 → script B
- 04:00 → script C

Men här uppstår ett problem. onödig väntetid. 

### DAG: Directed Acyclic Graph:

DAG = en graf med riktade pilar och utan cykler.

DAG beskriver hur tasks i en pipeline hänger ihop och i vilken riktning de ska köras, utan att skapa loopar.

I en DAG är riktningen viktig eftersom den beskriver beroendet.Riktningen berättar alltså vilket steg som kommer före vilket annat steg.

**Cyclic betyder:**något som bildar en cykel/loop. **Acyclic betyder:**utan cykler

Alltså: Det får inte finnas en loop i vår DAG.

En DAG är en karta över vår pipeline. Kartan berättar:

- Vad ska göras?
- Vilka tasks finns?
- Vilka tasks måste vänta på andra?
- Vilka kan köras samtidigt?
- När får nästa task starta?

I Airflow brukar vi prata om tasks. Så du kan tänka: Task = ett arbete som Airflow ska utföra.

Airflow använder DAG. En data pipeline består ofta av olika steg som har beroenden

Med DAG säger vi istället:

Kör B och C när deras dependencies är uppfyllda. När de steg som D behöver är färdiga, kör D.

**Upstream och downstream**

Nu kommer två ord som är väldigt viktiga när du arbetar med Airflow:

- Upstream : En task som kommer före en annan task.

- Downstream : En task som kommer efter en annan task.

### Airflow + Docker :

Airflow består av flera komponenter.

**Docker Compose** Docker Compose används när vi har flera containers/services som ska köras tillsammans.

Istället för att manuellt skriva massor av Docker-kommandon kan vi beskriva systemet i en fil. vanligtvis `docker-compose.yml` eller `compose.yml`.

Den säger ungefär: "Jag behöver dessa services, dessa inställningar, dessa volymer och dessa beroenden."

`docker compose up airflow-init` för att initiera Airflow, och senare använder vi Compose för att starta systemet.

**Varför behöver Airflow PostgreSQL?**

"Varför behöver Airflow en databas? Jag trodde Airflow var själva data-pipelinen?"

Airflow behöver själv lagra information om vad som händer.

Till exempel behöver Airflow kunna hålla reda på saker som:
- DAG
- Task
- Task status
- Körningar
- Historik
- Metadata

Därför använder Airflow en metadata-databas. Airflows metadata-databas är till för Airflow.

**Metadata** är information om systemets data och körningar. I Airflow kan det exempelvis handla om:
- Vilken DAG?
- Vilken task?
- När kördes den?
- Lyckades den?
- Misslyckades den?
- Vilken körning tillhör den?

Det är den typen av information Airflow behöver hålla reda på.

- **Airflow Webserver** Airflow Webserver är den del av Airflow som ger dig ett webbgränssnitt (UI) där du kan se och övervaka dina pipelines.webservern gör att man kan koppla upp sig och se vad som händer i Airflow.
- **Scheduler** hanterar schemaläggning och vilka tasks som ska köras.schemaläggare:Dess uppgift är att titta på dina DAG:ar och avgöra vilka tasks som är redo att köras.
- **CLI Command Line Interface** Alltså ett sätt att kommunicera med Airflow från terminalen.

"Airflow består av flera komponenter som kan köras som separata Docker services/containers och som tillsammans bildar Airflow-systemet."

Webserver är för dig som användare. Scheduler arbetar bakom kulisserna med Airflows körningar.

#### PostgreSQL: Airflows interna databas:

Airflow behöver själv en databas för att hålla reda på vad som händer i Airflow.

Men Airflow behöver information om sina egna körningar.

Till exempel:

- DAG: daily_sales
- Task: extract
- Status: success
- Start: 02:00
- Slut: 02:03

Det är metadata om Airflows arbete. I Docker Compose körs PostgreSQL som en egen service/container tillsammans med Airflows övriga komponenter.

**Volume**

Gör att databasinformationen kan finnas kvar mellan containeromstarter.

Viktigt! Använd inte Airflows interna metadata-databas som din vanliga applikationsdatabas.

- Vad är en volume?

En Docker Volume är ett sätt att lagra data utanför containerns livscykel. Volume ser till att PostgreSQL:s data kan överleva containeromstarter.

**Docker Compose-filen**

Service = definitionen av en körbar del i Compose.

Container = den faktiska körande instansen.

- environment variables : Det är variabler som skickas in till containern när den startas.
Exempel:

environment:

  AIRFLOW__CORE__...

  AIRFLOW__DATABASE__...

Alltså Environment variables används för att ge programmet konfigurationsinformation.

Airflow behöver veta olika saker om sin miljö.

Till exempel kan den behöva veta:

- vilken databas den ska använda
- hur systemet ska konfigureras
- olika inställningar för Airflow

Istället för att hårdkoda allt direkt i programkoden kan konfigurationen skickas in som environment variables.

Docker Compose → sätter upp Airflow-systemet

Airflow DAG → beskriver data-pipelinen

**Docker dependencies handlar om services. Airflow dependencies handlar om tasks i DAG:en.**

### Starta Airflow med Docker Compose

`docker compose up` up betyder ungefär: Starta services som finns definierade i Compose-filen.

steg för steg:

- docker compose up     
- Docker läser compose-filen     
- Ser vilka services som behövs     
- Hämtar images om de saknas
- Skapar containers  
- Startar services

`docker compose up airflow-init` Vi startar alltså inte hela Airflow-systemet direkt.

Vi startar en särskild service som heter: `airflow-init` initialisering. Alltså: Gör det som behöver göras för att förbereda Airflow innan det körs normalt.
 
`docker compose up -d` Starta services i bakgrunden

**Airflow gör inte bara jobbet — det hjälper dig också att se hur jobbet gick.**

#### En viktig skillnad

Just nu ska du skilja på:

Pipeline = hela processen  Läs → Rensa → Spara

Task = ett enskilt steg  läs eller Rensa eller spara.

Men i en riktig data pipeline räcker det inte att bara ha flera tasks. Vi måste också tala om vilken task som ska göras före vilken annan.

### Hur skriver man en DAG i Python?

man kan definiera en DAG med en särskild Python-dekorator: @dag

när vi skriver:

@dag
`def my_first_dag():` så använder Airflow funktionen som definitionen av vår DAG.

en DAG-definition innehåller bland annat information om när den ska starta och hur ofta den ska köras.

`@dag` → berättar för Airflow att funktionen representerar en DAG.

**DAG:en behöver information om när den ska köras**

DAG

 │
 ├── start_date (när börjar schemat gälla?)

 │
 └── schedule (hur ofta ska den köras?)

Schedule inte själva tasken .Det är instruktionen till Airflow om när DAG:en ska triggas/köras enligt schemat.

`@dag `→ detta är en Airflow-DAG.

`start_date` → när schemat börjar gälla.

`schedule` → hur ofta DAG:en ska köras.

**Vad är cron?**

Cron är ett sätt att beskriva ett tidsschema. kan man använda en cron-sträng som Airflow kan tolka.

**Vad händer om en körning missas?**

Om `catchup = true` Då kan Airflow försöka köra ikapp de körningar som missades.

Alltså:

10:25
 
upptäcker:
1. 10:10 missad
2. 10:15 missad
3. 10:20 missad

kör de missade

`catchup = False` Strunta i det som missades. Börja från där vi är nu.

**Hur kopplar vi ihop tasks i Airflow?**

setup >> extract >> load

Vad betyder **>>**? Det betyder: Den vänstra tasken ska komma före den högra tasken. Med **>>** berättar vi alltså för Airflow ordningen mellan tasks.

detta är viktigt för att Airflow måste veta beroendet mellan tasks. **>>** betyder inte: "Kör Python-funktionen direkt." Det betyder snarare:"Skapa en relation/beroende mellan dessa Airflow-tasks."

**Vad är en Operator?**

"Det här talar om för Airflow vilken typ av arbete tasken ska utföra."

Vi vet nu att en task är ett steg som Airflow ska utföra.Men Airflow behöver också veta: Vad ska den här tasken faktiskt göra? 

En operator beskriver alltså vilken typ av arbete tasken ska utföra.

1. **BashOperator :** Den används när tasken ska köra ett Bash-/terminalkommando. Till exempel `python script.py` Normalt kör du kommandot i terminalen.Med BashOperator kan Airflow göra det åt dig som en task
2. **PythonOperator :** Den används när tasken ska köra en Python-funktion.

- ETL = vad vi gör med datan.
- Airflow = hur vi orkestrerar och övervakar stegen.

#### Docker + Airflow

Airflow behöver en miljö där alla dess delar och beroenden finns.Docker gör det möjligt att skapa en sådan miljö i containers.

Airflow kan köras i en Docker-miljö, och Docker Compose används för att hantera miljön.

Hur ser en DAG ut i Python-koden?

`@dag`

`def my_dag():`

    task1 = ...
    task2 = ...
    task3 = ...

    task1 >> task2 >> task3`

`@task`
`def extract():`
    `print("Hämtar data")`    Här säger vi i princip:"Airflow, behandla den här funktionen som en task."

- @dag = hela arbetsflödet.
- @task = ett arbete/steg inne i arbetsflödet.   

"@task är ett enklare sätt att skapa tasks i en DAG, särskilt när tasken består av en Python-funktion." @task används ofta för att göra Python-funktioner till Airflow-tasks.

#### XCom – kommunikation mellan tasks

XCom gör det möjligt för tasks i Airflow att kommunicera och dela information.

Så XCom är inte bara något "osynligt" mellan tasks — Airflow kan också visa den information som har skickats.